## Obtención de Reparto y Palabras Clave

Una vez que hemos limpiado y filtrado el dataset de películas, descargamos información complementaria de TMDB: el **reparto** (director y actores principales) y las **palabras clave** asociadas a cada película.

Esta información enriquece el perfil de contenido de cada película y es fundamental para el recomendador basado en contenido, que utiliza embeddings de texto para calcular similitudes:

- **Director:** define el estilo cinematográfico de la película
- **Actores principales:** influyen directamente en la percepción del público
- **Palabras clave:** capturan temáticas, atmósferas y elementos narrativos específicos

In [2]:
import ast
import pandas as pd
import numpy as np
import requests
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load the environment variables .env
load_dotenv()

True

Cargamos los IDs de las películas seleccionadas en el notebook anterior.

In [3]:
ids = pd.read_csv('../data/processed/clean_movies_ids.csv')
# Rename back to match movies_metadata columns for merging
ids = ids.rename(columns={'movielens_id': 'movieId', 'tmdb_id': 'id'})
ids.head()

,movieId,id
0,58559,155
1,5618,129
2,7153,122
3,3147,497
4,109487,157336


Creamos una función genérica para consumir la API de TMDB y funciones específicas para los endpoints de créditos (`/credits`) y palabras clave (`/keywords`).

In [4]:
def fetch_tmdb_data(movie_id, api_key, endpoint=""):
    """Generic function to fetch any TMDB data for a movie"""
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/{endpoint}?api_key={api_key}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:  # Too Many Requests
            # Sleep for a bit if we hit rate limits
            time.sleep(2)
            return fetch_tmdb_data(movie_id, api_key, endpoint)  # Simple retry
        else:
            print(f"Error: HTTP {response.status_code} for movie {movie_id}")
            return None
    except Exception as e:
        print(f"Error fetching data for movie {movie_id}: {e}")
        return None

In [5]:
# Create endpoint-specific functions that use the global one
def fetch_credits(movie_id, api_key):
    return fetch_tmdb_data(movie_id, api_key, "credits")

def fetch_keywords(movie_id, api_key):
    return fetch_tmdb_data(movie_id, api_key, "keywords")

In [6]:
def fetch_movies(ids, api_key, fetch_func, max_workers=10, delay=0.05):
    """
    Function to fetch movie data for multiple movie IDs concurrently
    
    Args:
        ids: List of movie IDs
        api_key: TMDB API key
        fetch_func: Function to fetch specific data (e.g., fetch_credits)
        max_workers: Max number of concurrent requests
        delay: Small delay between requests to be nice to API
        
    Returns:
        (successful_results, failed_ids)
    """
    successful = []
    failed = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_func, movie_id, api_key): movie_id for movie_id in ids}
        
        for future in as_completed(futures):
            movie_id = futures[future]
            result = future.result()
            
            if result:
                successful.append(result)
            else:
                failed.append(movie_id)
            
            # Small delay to avoid hammering the API
            time.sleep(delay)
    
    return successful, failed

In [7]:
api_key = os.getenv('tmdb_api_key')
movie_ids = ids['id']

Una vez creadas las funciones, iniciamos la descarga de **palabras clave** para las 5,000 películas usando peticiones concurrentes para reducir el tiempo total.

In [8]:
# Fetching keywords
keywords, failed_keywords = fetch_movies(movie_ids, api_key, fetch_keywords)
print(f"Fetched {len(keywords)} movie keywords, failed {len(failed_keywords)}")

Fetched 5000 movie keywords, failed 0


#### Palabras Clave

In [9]:
keywords = pd.DataFrame(keywords)
keywords.head()

,id,keywords
0,157336,"[{'id': 10084, 'name': 'rescue'}, {'id': 2964,..."
1,122,"[{'id': 6092, 'name': 'army'}, {'id': 818, 'na..."
2,27205,"[{'id': 10364, 'name': 'mission'}, {'id': 1566..."
3,155,"[{'id': 4426, 'name': 'sadism'}, {'id': 4630, ..."
4,372058,"[{'id': 6270, 'name': 'high school'}, {'id': 4..."


Descargamos los **créditos** (reparto y equipo) de cada película.

In [10]:
# Fetching credits
credits, failed_credits = fetch_movies(movie_ids, api_key, fetch_credits)
print(f"Fetched {len(credits)} movie credits, failed {len(failed_credits)}")

Fetched 5000 movie credits, failed 0


#### Créditos

In [11]:
credits = pd.DataFrame(credits)
credits.head()

,id,cast,crew
0,497,"[{'adult': False, 'gender': 2, 'id': 31, 'know...","[{'adult': False, 'gender': 2, 'id': 6800, 'kn..."
1,155,"[{'adult': False, 'gender': 2, 'id': 3894, 'kn...","[{'adult': False, 'gender': 2, 'id': 3904, 'kn..."
2,496243,"[{'adult': False, 'gender': 2, 'id': 20738, 'k...","[{'adult': False, 'gender': 0, 'id': 3111262, ..."
3,120,"[{'adult': False, 'gender': 2, 'id': 109, 'kno...","[{'adult': False, 'gender': 2, 'id': 108, 'kno..."
4,27205,"[{'adult': False, 'gender': 2, 'id': 6193, 'kn...","[{'adult': False, 'gender': 2, 'id': 525, 'kno..."


Verificamos que no haya IDs duplicados en los DataFrames de créditos y palabras clave antes de continuar con el procesamiento.

In [12]:
# Ensure we do not have duplicated rows
credits[credits['id'].duplicated()].shape[0], keywords[keywords['id'].duplicated()].shape[0]

(0, 0)

## Extracción de Director y Actores

Del campo `crew` extraemos únicamente el **director**, ya que es la persona que más influye en el estilo de la película.

Del campo `cast` seleccionamos los **3 actores principales** (los primeros en la lista de créditos), ya que los actores secundarios y papeles menores no suelen influir en la percepción general del público.

In [13]:
# Convertfrom JSON to a Pyhton Object
credits['crew'] = credits['crew'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
credits['cast'] = credits['cast'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

In [14]:
# Extract the director from the crew
def get_director(x):
    for item in x:
        if item['job'] == 'Director':
            return item['name']
    return np.nan

In [15]:
credits['director'] = credits['crew'].apply(lambda x: get_director(x))
# Get the top 3 actors from the cast
credits['cast'] = credits['cast'].apply(lambda x: x[:3] if len(x) >=3 else x).apply(lambda x: [item['name'] for item in x])

In [16]:
credits.head()

,id,cast,crew,director
0,497,"[Tom Hanks, David Morse, Bonnie Hunt]","[{'adult': False, 'gender': 2, 'id': 6800, 'kn...",Frank Darabont
1,155,"[Christian Bale, Heath Ledger, Aaron Eckhart]","[{'adult': False, 'gender': 2, 'id': 3904, 'kn...",Christopher Nolan
2,496243,"[Song Kang-ho, Lee Sun-kyun, Cho Yeo-jeong]","[{'adult': False, 'gender': 0, 'id': 3111262, ...",Bong Joon Ho
3,120,"[Elijah Wood, Ian McKellen, Viggo Mortensen]","[{'adult': False, 'gender': 2, 'id': 108, 'kno...",Peter Jackson
4,27205,"[Leonardo DiCaprio, Joseph Gordon-Levitt, Ken ...","[{'adult': False, 'gender': 2, 'id': 525, 'kno...",Christopher Nolan


Eliminamos la columna `crew` ya que ya extrajimos la información relevante (el director).

In [17]:
# Delete the crew column
credits.drop(columns='crew', inplace=True)

# Nan if the film do not have information about the actors
credits['cast'] = credits['cast'].apply(lambda x: np.nan if not x else x)
credits.isna().sum()

id           0
cast        17
director     1
dtype: int64

Verificamos el estado del DataFrame de créditos antes de continuar.

## Extracción de Palabras Clave

Procesamos la columna de palabras clave: cada película tiene una lista de diccionarios con los campos `id` y `name`. Extraemos solo los nombres para crear una lista limpia de keywords.

In [18]:
keywords.iloc[0]['keywords']

[{'id': 10084, 'name': 'rescue'},
 {'id': 2964, 'name': 'future'},
 {'id': 1612, 'name': 'spacecraft'},
 {'id': 4776, 'name': 'race against time'},
 {'id': 310, 'name': 'artificial intelligence (a.i.)'},
 {'id': 1432, 'name': 'nasa'},
 {'id': 1521, 'name': 'time warp'},
 {'id': 4565, 'name': 'dystopia'},
 {'id': 1963, 'name': 'expedition'},
 {'id': 3801, 'name': 'space travel'},
 {'id': 3417, 'name': 'wormhole'},
 {'id': 4337, 'name': 'famine'},
 {'id': 6567, 'name': 'hibernation'},
 {'id': 4380, 'name': 'black hole'},
 {'id': 8056, 'name': 'quantum mechanics'},
 {'id': 10235, 'name': 'family relationships'},
 {'id': 9882, 'name': 'space'},
 {'id': 14544, 'name': 'robot'},
 {'id': 14626, 'name': 'astronaut'},
 {'id': 14760, 'name': 'scientist'},
 {'id': 33479, 'name': 'single father'},
 {'id': 154846, 'name': 'farmer'},
 {'id': 156039, 'name': 'space station'},
 {'id': 195114, 'name': 'space adventure'},
 {'id': 208757, 'name': 'time paradox'},
 {'id': 220800, 'name': 'time-manipulatio

La columna `keywords` contiene listas de diccionarios con la estructura `{'id': ..., 'name': ...}`. Extraemos únicamente los nombres y marcamos como nulo si una película no tiene keywords.

In [19]:
keywords['keywords'] = keywords['keywords'].apply(lambda x: json.loads(x) if isinstance(x, str) else x) \
                        .apply(lambda x: [item['name'] for item in x]) \
                        .apply(lambda x: np.nan if not x else x)
keywords.head()

,id,keywords
0,157336,"[rescue, future, spacecraft, race against time..."
1,122,"[army, based on novel or book, elves, dwarf, m..."
2,27205,"[mission, dreams, kidnapping, spy, allegory, i..."
3,155,"[sadism, chaos, secret identity, crime fighter..."
4,372058,"[high school, race against time, dreams, after..."


In [20]:
keywords.isna().sum()

id            0
keywords    189
dtype: int64

Las películas sin palabras clave son aquellas para las que TMDB no tiene esta información. Estas películas seguirán en el dataset; el campo quedará vacío y se manejará como texto vacío al generar los embeddings del recomendador basado en contenido.

## Integración de Datos Adicionales

Combinamos los metadatos base (del notebook anterior) con el director, actores y palabras clave descargados. También calculamos el score de calidad de IMDb recalibrado para este catálogo.

In [21]:
mdf = pd.read_parquet('../Data/Raw/movies_metadata.parquet')
# Select only the relevant columns
mdf = mdf[['movieId', 'id', 'title', 'genres', 'overview', 
           'runtime', 'release_date', 'tagline', 'vote_count', 
           'vote_average', 'poster_path', 'backdrop_path']]
# Select the filtered movies
mdf = mdf[mdf['id'].isin(ids['id'])]

In [22]:
# Merge the DataFrames
df = pd.merge(pd.merge(mdf, credits, on='id'), keywords, on='id')
df.head().transpose()

,0,1,2,3,4
movieId,10,6,1,2,11
id,710,949,862,8844,9087
title,GoldenEye,Heat,Toy Story,Jumanji,The American President
genres,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...","[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...","[{'id': 10751, 'name': 'Family'}, {'id': 35, '...","[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n..."
overview,When a powerful satellite system falls into th...,Obsessive master thief Neil McCauley leads a t...,"Led by Woody, Andy's toys live happily in his ...",When siblings Judy and Peter discover an encha...,"Widowed U.S. president Andrew Shepherd, one of..."
runtime,130,170,81,104,113
release_date,1995-11-16,1995-12-15,1995-11-22,1995-12-15,1995-11-17
tagline,No limits. No fears. No substitutes.,A Los Angeles crime saga.,The adventure takes off when toys come to life!,It's a jungle in here.,Why can't the most powerful man in the world h...
vote_count,4278,8224,19723,11207,776
vote_average,6.9,7.931,7.971,7.2,6.53


Añadimos el score de calidad de IMDb (Weighted Rating), que ya calculamos en el notebook 3 sobre el catálogo filtrado. Este score será útil como señal de popularidad en el sistema de recomendación.

In [25]:
m = df['vote_count'].quantile(0.9)  # percentil 90, igual que en Notebook 3
C = df['vote_average'].mean()

def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v / (v + m) * R) + (m / (v + m) * C)

In [26]:
df['score'] = df.apply(weighted_rating, axis=1)

In [29]:
df.sort_values(by='score', ascending=False).head(20)[['title', 'score', 'vote_count', 'vote_average']]

,title,score,vote_count,vote_average
1293,The Dark Knight,8.297891,35465,8.528
2188,Interstellar,8.268127,39350,8.470
684,The Lord of the Rings: The Return of the King,8.210106,26305,8.496
1612,Inception,8.207556,38982,8.400
318,Fight Club,8.171999,31765,8.400
496,The Lord of the Rings: The Fellowship of the Ring,8.167058,27303,8.431
3832,Parasite,8.147664,20399,8.494
558,Spirited Away,8.144856,18149,8.533
337,The Green Mile,8.137981,19057,8.505
586,The Lord of the Rings: The Two Towers,8.123974,23682,8.415


Convertimos la columna `genres` de JSON a lista de Python para facilitar el procesamiento posterior. También rellenamos los valores nulos en `tagline` con cadena vacía.

In [30]:
# Extract genres
df['genres'] = df['genres'].apply(lambda val: [item['name'] for item in val])

# Delete the Nan in the tagline column
df['tagline'] = df['tagline'].fillna('')

Verificamos que no haya títulos o IDs duplicados en el dataset final.

In [31]:
# Verify no duplicated titles (dedup was done in Notebook 3)
rt = df['title'].duplicated().sum()
rtmdb = df['id'].duplicated().sum()
rid = df['movieId'].duplicated().sum()

print(f'There are {rt} duplicated titles, {rtmdb} duplicated TMDB ids, and {rid} duplicated movie ids.')
assert rt == 0, f'Expected 0 duplicated titles, got {rt}. Dedup should have been done in Notebook 3.'

There are 0 duplicated titles, 0 duplicated TMDB ids, and 0 duplicated movie ids.


La deduplicación se realizó en el Notebook 3 (antes de seleccionar las 5,000 películas), por lo que no deberían existir duplicados. El `assert` confirmará esto.

In [32]:
df_final = df.copy()
df_final.shape

(5000, 16)

In [33]:
rt = df_final['title'].duplicated().sum()
rtmdb = df_final['id'].duplicated().sum()
rid = df_final['movieId'].duplicated().sum()

print(f'There are {rt} duplicated titles, {rtmdb} duplicated TMDB ids, and {rid} duplicated movie ids.')

There are 0 duplicated titles, 0 duplicated TMDB ids, and 0 duplicated movie ids.


In [34]:
df_final['movieId'] = df_final['movieId'].astype(int)
df_final.head()

,movieId,id,title,genres,overview,runtime,release_date,tagline,vote_count,vote_average,poster_path,backdrop_path,cast,director,keywords,score
0,10,710,GoldenEye,"[Adventure, Action, Thriller]",When a powerful satellite system falls into th...,130,1995-11-16,No limits. No fears. No substitutes.,4278,6.900,/z0ljRnNxIO7CRBhLEO0DvLgAFPR.jpg,/fIWsCpYR9iGDMSbMTSAzy8L7Kg5.jpg,"[Pierce Brosnan, Sean Bean, Izabella Scorupco]",Martin Campbell,"[computer virus, cuba, falsely accused, secret...",7.076005
1,6,949,Heat,"[Crime, Drama, Action]",Obsessive master thief Neil McCauley leads a t...,170,1995-12-15,A Los Angeles crime saga.,8224,7.931,/umSVjVdbVwtx5ryCA2QXL44Durm.jpg,/xKsnZDERG1dk95wuZ5q9iks3OL3.jpg,"[Al Pacino, Robert De Niro, Val Kilmer]",Michael Mann,"[robbery, chase, obsession, detective, heist, ...",7.577608
2,1,862,Toy Story,"[Family, Comedy, Animation, Adventure]","Led by Woody, Andy's toys live happily in his ...",81,1995-11-22,The adventure takes off when toys come to life!,19723,7.971,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg,"[Tom Hanks, Tim Allen, Don Rickles]",John Lasseter,"[rescue, friendship, mission, jealousy, villai...",7.757213
3,2,8844,Jumanji,"[Adventure, Fantasy, Family]",When siblings Judy and Peter discover an encha...,104,1995-12-15,It's a jungle in here.,11207,7.200,/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,/qSxeCfWUUyht9hZgaaYmtPtTkw2.jpg,"[Robin Williams, Kirsten Dunst, Bradley Pierce]",Joe Johnston,"[giant insect, board game, disappearance, jung...",7.191806
4,11,9087,The American President,"[Drama, Romance]","Widowed U.S. president Andrew Shepherd, one of...",113,1995-11-17,Why can't the most powerful man in the world h...,776,6.530,/yObOAYFIHXHkFPQ3jhgkN2ezaD.jpg,/62BnXyJtVEq4WKNSpnPG7QPYYDI.jpg,"[Michael Douglas, Annette Bening, Martin Sheen]",Rob Reiner,"[new love, usa president, the white house, hol...",7.116798


## Asignación de `movie_id` Interno

Asignamos un índice secuencial 0-based (`movie_id`) para uso interno en el pipeline:

- **`movie_id`**: índice 0-based para indexar la matriz de similitud coseno
- **`movielens_id`**: ID original de MovieLens (para trazabilidad con el dataset de ratings)
- **`tmdb_id`**: ID de TMDB (para consultas a la API y para el dataset Survey)

Esta convención de nombres se mantiene en todos los notebooks y archivos del pipeline.

In [35]:
df_final.rename(columns={
    'movieId': 'movielens_id',
    'id': 'tmdb_id'
}, inplace=True)
df_final['movie_id'] = range(df_final.shape[0])
df_final.head()

,movielens_id,tmdb_id,title,genres,overview,runtime,release_date,tagline,vote_count,vote_average,poster_path,backdrop_path,cast,director,keywords,score,movie_id
0,10,710,GoldenEye,"[Adventure, Action, Thriller]",When a powerful satellite system falls into th...,130,1995-11-16,No limits. No fears. No substitutes.,4278,6.900,/z0ljRnNxIO7CRBhLEO0DvLgAFPR.jpg,/fIWsCpYR9iGDMSbMTSAzy8L7Kg5.jpg,"[Pierce Brosnan, Sean Bean, Izabella Scorupco]",Martin Campbell,"[computer virus, cuba, falsely accused, secret...",7.076005,0
1,6,949,Heat,"[Crime, Drama, Action]",Obsessive master thief Neil McCauley leads a t...,170,1995-12-15,A Los Angeles crime saga.,8224,7.931,/umSVjVdbVwtx5ryCA2QXL44Durm.jpg,/xKsnZDERG1dk95wuZ5q9iks3OL3.jpg,"[Al Pacino, Robert De Niro, Val Kilmer]",Michael Mann,"[robbery, chase, obsession, detective, heist, ...",7.577608,1
2,1,862,Toy Story,"[Family, Comedy, Animation, Adventure]","Led by Woody, Andy's toys live happily in his ...",81,1995-11-22,The adventure takes off when toys come to life!,19723,7.971,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg,"[Tom Hanks, Tim Allen, Don Rickles]",John Lasseter,"[rescue, friendship, mission, jealousy, villai...",7.757213,2
3,2,8844,Jumanji,"[Adventure, Fantasy, Family]",When siblings Judy and Peter discover an encha...,104,1995-12-15,It's a jungle in here.,11207,7.200,/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,/qSxeCfWUUyht9hZgaaYmtPtTkw2.jpg,"[Robin Williams, Kirsten Dunst, Bradley Pierce]",Joe Johnston,"[giant insect, board game, disappearance, jung...",7.191806,3
4,11,9087,The American President,"[Drama, Romance]","Widowed U.S. president Andrew Shepherd, one of...",113,1995-11-17,Why can't the most powerful man in the world h...,776,6.530,/yObOAYFIHXHkFPQ3jhgkN2ezaD.jpg,/62BnXyJtVEq4WKNSpnPG7QPYYDI.jpg,"[Michael Douglas, Annette Bening, Martin Sheen]",Rob Reiner,"[new love, usa president, the white house, hol...",7.116798,4


## Guardado del Dataset Final

Guardamos el dataset enriquecido como `movies_final.csv`. Este archivo es el punto de partida para todos los notebooks posteriores: filtrado de ratings, extracción de personalidad, detalles en español y construcción de la matriz de similitud coseno.

In [36]:
# Save it as the final dataset
df_final.to_csv('../data/processed/movies_final.csv', index=False)

## Resumen del Notebook

En este notebook se enriqueció el catálogo de películas con información de créditos y palabras clave de TMDB.

**Acciones realizadas:**
- Descarga de créditos (director, actores) y palabras clave para las 5,000 películas
- Extracción del director y los 3 actores principales de cada película
- Verificación de ausencia de duplicados (garantizada por el Notebook 3)
- Asignación de índice interno `movie_id` (0-based) para el pipeline

**Resultado:** Dataset con las 5,000 películas enriquecidas con director, reparto y keywords

**Output generado:** `ml/data/processed/movies_final.csv`